In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# превращает корпус текстов в матрицу подсчёта токенов
vectorizer = CountVectorizer(
  ngram_range=(1, 2), # (1, 1) - BoW
  #  (1, 2) - unigrams+bigrams (2, 2) - только bigrams

  max_features=10000, # 10000 самых частых слов

  min_df=2, # min freq == 2

  max_df=0.8, # игнорировать слова
  # которые больше чем в 80 проц документов

  lowercase=True # всё к нижнему регистру
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
  ngram_range=(1, 2),
  max_features=10000,
  sublinear_tf=True,
  # применяет 1+log() вместо чистого tf(сглаживания)
  norm='l2' # нормирует вектор каждого документа до единичной l_2 - длины
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# ==========================================
# 1. Синтетический датасет банковских жалоб
# ==========================================
data = {
    "text": [
        "заблокировали дебетовую карту без предупреждения",
        "не могу перевести деньги через приложение",
        "приложение вылетает при входе в личный кабинет",
        "подозрительное списание средств со счета мошенники",
        "карта заблокирована банком как разблокировать",
        "быстрый и удобный перевод денег на карту другого банка",
        "ужасный сервис поддержка не отвечает целый день",
        "украли деньги с карты списали все средства",
        "не работает перевод по номеру телефона",
        "требую вернуть деньги заблокировали счет мошенничество"
    ] * 50,  # Увеличим выборку для стабильности
    "category": [
        "Cards", "Transfers", "Service", "Fraud", "Cards",
        "Transfers", "Service", "Fraud", "Transfers", "Fraud"
    ] * 50
}

df = pd.DataFrame(data)

# Разделение на train и test
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["category"], test_size=0.2, random_state=42, stratify=df["category"]
)

# ==========================================
# ЗАДАНИЕ 1: Bag-of-Words (BoW)
# ==========================================
# TODO 1:
# 1. Создайте CountVectorizer с ngram_range=(1,1)
# 2. Обучите и трансформируйте X_train через fit_transform
# 3. Трансформируйте X_test через transform
# 4. Выведите размер матрицы X_train_bow (shape) и первые 10 слов из словаря (get_feature_names_out)

BoW = CountVectorizer(
    ngram_range=(1, 1),
    lowercase=True
)

X_train_bow = BoW.fit_transform(X_train)
X_test_bow = BoW.transform(X_test)

print(f"1. BoW shape: {X_train_bow.shape}")
print(*BoW.get_feature_names_out()[:10], '\n')

# ==========================================
# ЗАДАНИЕ 2: N-Grams (Unigrams + Bigrams)
# ==========================================
# TODO 2:
# 1. Создайте CountVectorizer с ngram_range=(1,2)
# 2. Получите матрицы X_train_ngram и X_test_ngram
# 3. Сравните размер словаря N-grams с обычным BoW из Задания 1 (на сколько выросло число признаков?)

ngram_vec = CountVectorizer(
    ngram_range=(1, 2),
    lowercase=True
)
X_train_ngram = ngram_vec.fit_transform(X_train)
X_test_ngram = ngram_vec.transform(X_test)

print(f"2. Ngram Shape {X_train_ngram.shape}")
print("Количество признаков выросло почти в 2 раза\n")

# ==========================================
# ЗАДАНИЕ 3: TF-IDF + Классификация
# ==========================================
# TODO 3:
# 1. Создайте TfidfVectorizer с ngram_range=(1,2), sublinear_tf=True
# 2. Получите матрицы X_train_tfidf и X_test_tfidf
# 3. Обучите модель LogisticRegression(max_iter=1000) на X_train_tfidf, y_train
# 4. Получите предсказания на X_test_tfidf и выведите classification_report

tfidf_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

clf = LogisticRegression()
clf.fit(X_train_tfidf, y_train)
preds = clf.predict(X_test_tfidf)
print(classification_report(y_test, preds))

1. BoW shape: (400, 53)
банка банком без быстрый вернуть все входе вылетает дебетовую денег 

2. Ngram Shape (400, 104)
Количество признаков выросло почти в 2 раза

              precision    recall  f1-score   support

       Cards       1.00      1.00      1.00        20
       Fraud       1.00      1.00      1.00        30
     Service       1.00      1.00      1.00        20
   Transfers       1.00      1.00      1.00        30

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

